# Attention ablations.

Runs the full study end to end on a Colab GPU.

Runtime type:T4 GPU in Google Colab

The whole study is roughly 10 configurations x 3 seeds. On a T4 with 20,000 training
examples it should take approx 45 to 90 minutes. Start with the `small` run if you
want to see the shape of the results first.

## 1. Install and clone

In [ ]:
!git clone https://github.com/StochasticSpirit/attention-ablations.git
%cd attention-ablations
!pip install -q -r requirements.txt

## 2. Confirm the implementation is correct

These are the numerical equivalence tests against PyTorch's own attention. 

In [ ]:
!PYTHONPATH=src python -m pytest tests/ -v --tb=short

# 3. Check if GPU is being used

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 4. Offline smoke test

To check if the code pipeline runs. Does not require a GPU.

In [ ]:
!python scripts/run_ablations.py --synthetic --epochs 4 --seeds 2 \
    --patience 0 --d-model 32 --d-ff 64 --out results/smoke

## 5. Small run

Uses 1,000 training examples.

In [ ]:
!python scripts/run_ablations.py --n-train 1000 --n-val 500 --n-test 500 \
    --epochs 20 --seeds 3 --out results/small

## 6. Data-scaling sweep

Trains the model at 500, 1,000, 2,000, 5,000, 10,000 and 20,000 examples and fits tf-idf
baselines on the same subsets. Reports the crossover point: the training set size at which
the Transformer overtakes the linear model.

Validation and test sets are identical at every training ecamples size. The vocabulary is rebuilt from
each training subset to avoid data leakage.

In [ ]:
# A first look: 3 sizes, 1 seed.
!python scripts/run_scaling.py --sizes 500 2000 10000 \
    --epochs 20 --seeds 1 --batch-size 64 --out results/scaling-quick

In [ ]:
# Detailed sweep: 6 sizes, 3 seeds.
!python scripts/run_scaling.py --sizes 500 1000 2000 5000 10000 20000 \
    --epochs 30 --seeds 3 --batch-size 64 --out results/scaling

### The scaling curve and the crossover point

In [ ]:
from IPython.display import Image, display

display(Image("results/scaling/scaling_curve.png"))
display(Image("results/scaling/scaling_gap.png"))
print(open("results/scaling/scaling_table.md").read())

## 7. The full ablation run

Runs on 20,000 training examples, the full 25,000 example test set, 3 seeds per configuration.

In [ ]:
!python scripts/run_ablations.py --n-train 20000 --n-val 5000 --n-test 25000 \
    --epochs 30 --seeds 3 --batch-size 64 --out results/full

## 8. Ablation figures

In [ ]:
from IPython.display import Image, display
from pathlib import Path

run = Path("results/full")
for name in ["ablations.png", "training_curves.png", "attention_heads.png"]:
    path = run / name
    if path.exists():
        print(name)
        display(Image(str(path)))

## 9. Ablation table

In [ ]:
print(open("results/full/results_table.md").read())

## 10. Which tokens the model actually attends to
In the initial model notebook we found that heads focus on sentiment-bearing words. We recheck that claim here.

In [ ]:
print(open("results/full/attention_ranking.txt").read())